# 🏆 Best Model Publisher: IEEE Figures & Archive

**Objective:** Identify the absolute best checkpoint from all training phases (10-13), evaluate it on the independent test set, and generate high-resolution IEEE-style figures for publication.

**Features:**
- 🔍 **Recursive Scan**: Checks `checkpoints_phase10`, `11`, `12`, `13` and `h5_checkpoints`.
- 🧠 **Auto-Introspection**: Automatically detects model config (`d_model`, layers) from weights.
- 📊 **Publication Plots**: Generates 300 DPI Confusion Matrix & ROC Curve (Times New Roman font).
- 💾 **Archive**: Copies the best model and figures to `achieved/publication/`.

In [ ]:
# 1. Setup Environment
from google.colab import drive
import os, sys, subprocess, shutil

drive.mount('/content/drive')

REPO_DIR = '/content/phase2'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/nithin12342/phase2.git', REPO_DIR])
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'])

sys.path.insert(0, f"{REPO_DIR}/ml_pipeline/h5_omnifusion")

subprocess.run(['pip', 'install', 'torch', 'torchvision', 'torchaudio', 'h5py', 
                'pandas', 'scikit-learn', 'matplotlib', 'seaborn', 'tqdm', '--quiet'])

print('✅ Environment Ready')

In [ ]:
# 2. Configuration & Paths
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc, f1_score, accuracy_score, precision_score, recall_score, roc_auc_score
import pandas as pd
import glob
import time
from collections import OrderedDict

# Project Imports
from config.model_config import H5Config, ComputeTier
from src.models.h5_omnifusion import H5OmniFusion
from src.data.h5_dataset import create_h5_dataloaders_kfold

DATA_ROOT = "/content/drive/MyDrive/DAIC-WOZ_Datasets"
H5_DIR = f"{DATA_ROOT}/H5_OmniFusion_Output"
LABELS_CSV = f"{DATA_ROOT}/phase13_labels.csv" # Use merged labels if available
if not os.path.exists(LABELS_CSV):
    LABELS_CSV = f"{DATA_ROOT}/H5_OmniFusion_Output/all_labels.csv"

PUBLICATION_DIR = f"{DATA_ROOT}/achieved/publication"
os.makedirs(PUBLICATION_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"📂 Saving publication results to: {PUBLICATION_DIR}")

In [ ]:
# 3. Scan for Checkpoints
print("🔍 Scanning for checkpoints...")
search_dirs = [
    f"{DATA_ROOT}/checkpoints_phase13",
    f"{DATA_ROOT}/checkpoints_phase12",
    f"{DATA_ROOT}/checkpoints_phase11",
    f"{DATA_ROOT}/checkpoints_phase10_finetune",
    f"{DATA_ROOT}/h5_checkpoints"
]

checkpoint_files = []
for d in search_dirs:
    if os.path.exists(d):
        found = glob.glob(f"{d}/**/*best.pt", recursive=True)
        checkpoint_files.extend(found)

checkpoint_files = sorted(list(set(checkpoint_files)))
print(f"✅ Found {len(checkpoint_files)} 'best' checkpoints.")
for f in checkpoint_files[-5:]: print(f"  .../{os.path.basename(os.path.dirname(f))}/{os.path.basename(f)}")

In [ ]:
# 4. Define Publication Plotting Functions
def set_style():
    plt.style.use('seaborn-v0_8-paper')
    plt.rcParams.update({
        'font.family': 'serif',
        'font.serif': ['Times New Roman'],
        'font.size': 14,
        'axes.labelsize': 16,
        'axes.titlesize': 18,
        'xtick.labelsize': 14,
        'ytick.labelsize': 14,
        'legend.fontsize': 14,
        'figure.dpi': 300,
    })

def plot_confusion_matrix(y_true, y_pred, save_path):
    cm = confusion_matrix(y_true, y_pred)
    classes = ['Non-Depressed', 'Depressed']
    
    plt.figure(figsize=(8, 6))
    cm_sum = np.sum(cm, axis=1, keepdims=True)
    cm_perc = cm / cm_sum.astype(float) * 100
    
    annot = np.empty_like(cm).astype(str)
    nrows, ncols = cm.shape
    for i in range(nrows):
        for j in range(ncols):
            c = cm[i, j]
            p = cm_perc[i, j]
            if i == j:
                s = cm_sum[i]
                annot[i, j] = '%.1f%%\n%d/%d' % (p, c, s)
            elif c == 0:
                annot[i, j] = ''
            else:
                annot[i, j] = '%.1f%%\n%d' % (p, c)
                
    sns.heatmap(cm, annot=annot, fmt='', cmap='Blues', cbar=False,
                xticklabels=classes, yticklabels=classes,
                annot_kws={"size": 16, "weight": "bold"},
                linewidths=1, linecolor='black', clip_on=False)
    
    plt.title('Confusion Matrix', pad=20)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight')
    plt.show()
    print(f"✅ Saved CM to {save_path}")

def plot_roc_curve(y_true, y_probs, save_path):
    fpr, tpr, _ = roc_curve(y_true, y_probs)
    roc_auc = auc(fpr, tpr)
    
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic', pad=20)
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight')
    plt.show()
    print(f"✅ Saved ROC to {save_path}")
    return roc_auc

In [ ]:
# 5. Helper: Introspect & Evaluate
def to_device(data, device):
    if isinstance(data, torch.Tensor): return data.to(device)
    elif isinstance(data, dict): return {k: to_device(v, device) for k, v in data.items()}
    return data

def evaluate(ckpt_path, loader):
    # Load Config from weights
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    sd = ckpt.get('model_state_dict', ckpt.get('state_dict', ckpt))
    
    # Heuristic config detection
    fname = os.path.basename(ckpt_path).lower()
    tier = ComputeTier.MEDIUM
    if "nano" in fname: tier = ComputeTier.NANO
    elif "micro" in fname: tier = ComputeTier.MICRO
    
    config = H5Config.from_tier(tier)
    
    # Adjust dimensions if mismatch
    for k, v in sd.items():
        if 'audio_encoder.input_proj.weight' in k:
             config.d_model = v.shape[0]
             config.audio.backbone_dim = v.shape[1]
             break
    
    model = H5OmniFusion(config)
    
    # Clean state dict keys
    new_sd = OrderedDict()
    for k, v in sd.items():
        name = k[7:] if k.startswith('module.') else k
        new_sd[name] = v
    
    model.load_state_dict(new_sd, strict=False)
    model.to(DEVICE)
    model.eval()
    
    all_true, all_pred, all_probs = [], [], []
    with torch.no_grad():
        for batch in loader:
            input_keys = [k for k in batch.keys() if k not in ['label', 'labels', 'target', 'targets', 'participant_id']]
            inputs = to_device({k: batch[k] for k in input_keys}, DEVICE)
            
            if 'label' in batch: labels = batch['label']['binary']
            elif 'targets' in batch: labels = batch['targets']['binary']
            else: continue
            
            out = model(inputs)
            probs = out[0]['binary_prob'].cpu().numpy().flatten()
            preds = (probs >= 0.5).astype(int)
            
            all_true.extend(labels.numpy())
            all_pred.extend(preds)
            all_probs.extend(probs)
            
    return np.array(all_true), np.array(all_pred), np.array(all_probs)

In [ ]:
# 6. Find Best Model
_, _, test_loader = create_h5_dataloaders_kfold(
    h5_dir=H5_DIR, labels_csv=LABELS_CSV,
    batch_size=32, fold_idx=0, n_folds=5
)

results = []
print(f"🚀 Evaluating {len(checkpoint_files)} checkpoints on Fold 0 test set...")

best_score = -1
best_ckpt = None
best_metrics = None

for ckpt in checkpoint_files:
    try:
        yt, yp, yprob = evaluate(ckpt, test_loader)
        f1 = f1_score(yt, yp, zero_division=0)
        auc_score = roc_auc_score(yt, yprob) if len(np.unique(yt)) > 1 else 0.5
        acc = accuracy_score(yt, yp)
        
        score = f1 + auc_score # Simple optimization metric
        
        print(f"  {os.path.basename(ckpt)[:40]:<40} | F1: {f1:.4f} | AUC: {auc_score:.4f}")
        
        results.append({'ckpt': ckpt, 'f1': f1, 'auc': auc_score, 'acc': acc})
        
        if score > best_score:
            best_score = score
            best_ckpt = ckpt
            best_metrics = (yt, yp, yprob)
            
    except Exception as e:
        print(f"❌ Failed {os.path.basename(ckpt)}: {e}")

In [ ]:
# 7. Publish Results
if best_ckpt:
    print(f"\n🏆 BEST MODEL: {os.path.basename(best_ckpt)}")
    print(f"   F1: {f1_score(best_metrics[0], best_metrics[1]):.4f}")
    
    # Copy best model
    dest_pt = f"{PUBLICATION_DIR}/BEST_MODEL_{os.path.basename(best_ckpt)}"
    shutil.copy(best_ckpt, dest_pt)
    print(f"💾 Saved model to: {dest_pt}")
    
    # Generate Figures
    set_style()
    
    # Confusion Matrix
    plot_confusion_matrix(best_metrics[0], best_metrics[1], 
                          f"{PUBLICATION_DIR}/confusion_matrix_published.png")
    
    # ROC Curve
    plot_roc_curve(best_metrics[0], best_metrics[2], 
                   f"{PUBLICATION_DIR}/roc_curve_published.png")
    
    print("\n✨ PUBLICATION READY.")
else:
    print("❌ No models evaluated successfully.")